# Lab 10 — Advanced RAG: Hybrid Search + Cross-Encoder Reranking

Lab 4 showed the basic RAG pipeline: one embedding model, cosine similarity, done. In production, naive dense retrieval leaves 10–30% of queries on the table — especially ones involving rare keywords, acronyms, product names, or proper nouns. The industry's answer is a **two-stage retrieval** stack:

```
query ┬─→ [dense]   ─┐
       │              ├─→ merge top-K ─→ [cross-encoder reranker] ─→ top-N → LLM
       └─→ [BM25]    ─┘      (via RRF)
```

Three upgrades over vanilla RAG, each earned by empirical pain:

1. **BM25 keyword retrieval** — catches rare-term queries that dense embeddings sometimes miss
2. **Reciprocal Rank Fusion (RRF)** — the canonical way to blend heterogeneous score distributions
3. **Cross-encoder reranker** — a *second, slower, more accurate* model re-orders the top candidates

### Why each piece exists

- **BM25** ([Robertson & Zaragoza, 2009](https://www.staff.city.ac.uk/~sbrp622/papers/foundations_bm25_review.pdf)) is a 40-year-old TF-IDF descendant. In 2024, it's still competitive with dense retrieval — and *strictly better* on rare-token queries where embeddings' semantic smoothing hurts. Sparse retrieval's strength: it scores based on exact term overlap, weighted by how distinctive each term is.
- **RRF** ([Cormack et al., 2009](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf)) blends ranked lists by summing `1/(k + rank)` across retrievers. Simple, parameter-light, beats most learned fusion methods. Default in Weaviate, Elasticsearch, Vespa.
- **Cross-encoder reranker** is a model that takes both query AND passage together and outputs a single relevance score. Much slower than dense retrieval (no precomputed vectors), but far more accurate per pair. Used to re-score the top 20–100 candidates from first-stage retrieval. BAAI's [bge-reranker-base](https://huggingface.co/BAAI/bge-reranker-base) is the go-to open option; Cohere Rerank is the go-to paid option.

### References to read alongside

- **[BGE paper (Xiao et al., 2023)](https://arxiv.org/abs/2309.07597)** — includes the reranker family.
- **[Reciprocal Rank Fusion (Cormack et al., 2009)](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf)** — the paper. 4 pages. Read it.
- **[ColBERT (Khattab & Zaharia, 2020)](https://arxiv.org/abs/2004.12832)** — late-interaction, a cross-encoder/bi-encoder hybrid.
- **[MS MARCO passage ranking leaderboard](https://microsoft.github.io/msmarco/TREC-Deep-Learning)** — where all these retrieval models are benchmarked against each other.
- **[Vespa docs on hybrid search](https://docs.vespa.ai/en/tutorials/hybrid-search.html)** — the production playbook at one of the better open search platforms.
- **[OpenAI's cookbook on hybrid search](https://cookbook.openai.com/examples/hybrid_search_and_reranking)** — modern reference implementation.

---

## Step 1 — Set up corpus + dense baseline, find a dense-failure case

The best way to see why hybrid search matters is to **find a query where dense retrieval fails**. Our corpus will include a few entries with distinctive proper nouns / acronyms — the kind of content dense embeddings smooth away.

In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

device = torch.device('cuda')

# Corpus includes rare proper nouns that test embeddings' generalization
corpus = [
    "Our internal CI/CD system is called NIMBUS-OPS. It auto-deploys every merged PR to staging within 90 seconds.",
    "The customer support portal is built on ZephyrDesk v3. It handles 50,000 tickets per day at 99.95% SLA.",
    "We use a custom observability stack combining Prometheus, Grafana, and Loki for logs aggregation.",
    "All employee badges are managed by the HALCYON access-control system, replaced in 2023 from the previous Mercury platform.",
    "Our data warehouse runs on Snowflake. Daily ETL batches process ~2 TB into fact tables for the BI team.",
    "Production Kubernetes clusters are managed through a GitOps layer built on ArgoCD + Kustomize.",
    "Engineering onboarding takes two weeks and includes a rotation through each of the six platform teams.",
    "Incident response follows the PagerDuty + Slack + Zoom bridge pattern. Ops rotations are weekly, 6 engineers per tier.",
    "Our internal ML platform, codenamed PRISM, runs on Ray and supports distributed training up to 128 GPUs.",
    "Employee expenses go through Concur Expense. Approval chains are based on manager org hierarchy with a 72-hour SLA.",
]
print(f'{len(corpus)} documents in corpus')

# Use BGE-small as the dense embedder (same as Lab 4)
embed_tokenizer = AutoTokenizer.from_pretrained('BAAI/bge-small-en-v1.5')
embed_model = AutoModel.from_pretrained('BAAI/bge-small-en-v1.5').to(device)
embed_model.eval()

def embed(texts, max_len=256):
    enc = embed_tokenizer(texts, padding=True, truncation=True, max_length=max_len, return_tensors='pt').to(device)
    with torch.no_grad():
        out = embed_model(**enc)
    return F.normalize(out.last_hidden_state[:, 0], dim=-1)

passage_embs = embed(corpus)
print(f'Embedded into {passage_embs.shape[1]}-dim vectors')

/usr/local/lib/python3.11/dist-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


10 documents in corpus


tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

Embedded into 384-dim vectors


In [2]:
# Find a dense-retrieval failure case.
# Query contains a rare proper noun (HALCYON) that should clearly match doc 3.
# Dense embeddings tend to smooth away exact term identity in favor of general semantics — so they might
# retrieve a different access-control-ish doc first instead.

failure_query = 'Who provides our employee badge access system?'
q_vec = embed([failure_query])
scores = (q_vec @ passage_embs.T).squeeze(0)
dense_top1 = int(scores.argmax().item())

# Show top-3 for context
top_scores, top_idx = torch.topk(scores, k=3)
print(f'DENSE RETRIEVAL for query: {failure_query!r}\n')
for rank, (s, i) in enumerate(zip(top_scores.cpu().tolist(), top_idx.cpu().tolist()), 1):
    marker = ' <-- should be this one (HALCYON)' if i == 3 else ''
    print(f'  #{rank} [idx={i}, score={s:.3f}] {corpus[i][:100]}...{marker}')

TRUE_RELEVANT_IDX = 3  # The HALCYON doc
dense_failure = {
    'query': failure_query,
    'dense_top1_idx': dense_top1,
    'true_relevant_idx': TRUE_RELEVANT_IDX,
    'dense_top1_text': corpus[dense_top1],
}
if dense_top1 != TRUE_RELEVANT_IDX:
    print(f'\n DENSE MISSED: returned idx {dense_top1} instead of the true {TRUE_RELEVANT_IDX} (HALCYON doc).')
    print(' Rare proper nouns like HALCYON are exactly where keyword retrieval shines.')

DENSE RETRIEVAL for query: 'Who provides our employee badge access system?'

  #1 [idx=3, score=0.650] All employee badges are managed by the HALCYON access-control system, replaced in 2023 from the prev... <-- should be this one (HALCYON)
  #2 [idx=9, score=0.595] Employee expenses go through Concur Expense. Approval chains are based on manager org hierarchy with...
  #3 [idx=1, score=0.554] The customer support portal is built on ZephyrDesk v3. It handles 50,000 tickets per day at 99.95% S...


In [3]:
from preporato_labs import Lab
lab = Lab('advanced-rag')
lab.check(1)

OK — corpus: 10 items. Dense retrieved idx 3 CORRECTLY (truly relevant: idx 3)
  NOTE: dense got this one right. On larger corpora and rarer terms, dense often misses. Hybrid + rerank catch both cases.
STEP_PASSED


Step 1 Complete! Scroll down to continue...

True

## Step 2 — BM25 keyword retrieval from scratch

BM25 ranks documents by how *distinctively* their terms match the query. The formula for a single doc D and query q:

$$\text{BM25}(q, D) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{\text{tf}(t, D) \cdot (k_1 + 1)}{\text{tf}(t, D) + k_1 \cdot (1 - b + b \cdot \frac{|D|}{\bar{D}})}$$

Two knobs:
- `k_1 ≈ 1.5` — how quickly term-frequency scoring saturates
- `b ≈ 0.75` — how much to normalize for document length

The IDF (inverse document frequency) punishes common terms and rewards rare ones. That's exactly what makes BM25 win on queries like *"Who provides our HALCYON system?"* — HALCYON appears in exactly one doc, so matching it is a strong signal.

### Why implement it ourselves

`rank_bm25` is a one-line pip install. But reading this 20-line implementation is worth more than using the library — you see exactly how sparse retrieval ranks work.

### BM25 — the sparse baseline

Before neural retrieval was cheap, everyone ran **BM25**: a tuned TF-IDF variant that scores documents against a query using per-term inverse-document-frequency and per-document length normalization. The math fits in 20 lines; the gotcha is the **IDF smoothing** — Robertson-Sparck-Jones uses `log((N - df + 0.5) / (df + 0.5) + 1)` which keeps scores sensible even for terms that appear in every or no documents.

In [4]:
import math, re
from collections import Counter

def tokenize(text):
    return re.findall(r'[a-z0-9]+', text.lower())

class BM25:
    def __init__(self, corpus_tokens, k1=1.5, b=0.75):
        self.k1 = k1; self.b = b
        self.docs = corpus_tokens
        self.N = len(corpus_tokens)
        self.doc_lens = [len(d) for d in corpus_tokens]
        self.avgdl = sum(self.doc_lens) / self.N
        # Document-frequency per term
        df = Counter()
        for d in corpus_tokens:
            for term in set(d):
                df[term] += 1
        # IDF per term (Robertson-Sparck-Jones smoothing)
        self.idf = {t: math.log((self.N - v + 0.5) / (v + 0.5) + 1) for t, v in df.items()}
        # Per-doc term frequencies
        self.tf = [Counter(d) for d in corpus_tokens]

    def score(self, query_tokens):
        scores = [0.0] * self.N
        for i in range(self.N):
            norm_len = (1 - self.b) + self.b * self.doc_lens[i] / self.avgdl
            for t in query_tokens:
                if t not in self.idf:
                    continue
                tf = self.tf[i].get(t, 0)
                if tf == 0:
                    continue
                scores[i] += self.idf[t] * (tf * (self.k1 + 1)) / (tf + self.k1 * norm_len)
        return scores

### Retrieve the failure_query with BM25

BM25 rewards **literal term overlap**. If the failure_query shares rare technical terms with the HALCYON doc, BM25 will nail it. If the query uses synonyms the corpus doesn't, BM25 will miss — which is exactly the weakness a dense retriever fixes in step 3.

In [5]:
corpus_tokens = [tokenize(d) for d in corpus]
bm25 = BM25(corpus_tokens)

q_tokens = tokenize(failure_query)
bm25_scores = bm25.score(q_tokens)
bm25_top1_idx = max(range(len(bm25_scores)), key=lambda i: bm25_scores[i])

print(f'BM25 retrieval for query: {failure_query!r}')
ordered = sorted(range(len(bm25_scores)), key=lambda i: -bm25_scores[i])[:3]
for rank, i in enumerate(ordered, 1):
    marker = ' <-- HALCYON doc' if i == TRUE_RELEVANT_IDX else ''
    print(f'  #{rank} [idx={i}, score={bm25_scores[i]:.3f}] {corpus[i][:100]}...{marker}')

BM25 retrieval for query: 'Who provides our employee badge access system?'
  #1 [idx=3, score=4.759] All employee badges are managed by the HALCYON access-control system, replaced in 2023 from the prev... <-- HALCYON doc
  #2 [idx=0, score=2.461] Our internal CI/CD system is called NIMBUS-OPS. It auto-deploys every merged PR to staging within 90...
  #3 [idx=9, score=1.423] Employee expenses go through Concur Expense. Approval chains are based on manager org hierarchy with...


In [6]:
lab.check(2)

OK — BM25 retrieved the rare-keyword relevant doc (idx 3) where dense retrieval missed it
STEP_PASSED


Step 2 Complete! Scroll down to continue...

True

## Step 3 — Reciprocal Rank Fusion: combine dense + BM25

Two ranked lists, different score scales (cosine 0-1 vs BM25 unbounded). You can't just add their scores. **Reciprocal Rank Fusion** sidesteps the calibration problem:

$$\text{RRF}(d) = \sum_{r \in \text{rankers}} \frac{1}{k + \text{rank}_r(d)}$$

where `k ≈ 60` dampens the top-rank dominance. No calibration, no tuning per retriever. A document ranked 1st by both gets ~1/61 + 1/61 = 0.033; ranked 10th by both gets ~1/70 + 1/70 = 0.029. The monotonic difference is what matters.

This is the default fusion method in Elasticsearch's hybrid mode, Weaviate, Vespa, and most serious production RAG stacks.

In [9]:
def rrf_scores_of(rank_lists, k=60):
    """Each rank_list is an ordered list of doc indices, best first."""
    scores = [0.0] * len(corpus)
    for rl in rank_lists:
        for rank, doc_idx in enumerate(rl):
            scores[doc_idx] += 1.0 / (k + rank + 1)   # +1 because rank should be 1-indexed
    return scores

# Build ranked lists (best-first)
dense_scores_list = scores.cpu().tolist()
dense_ranking = sorted(range(len(corpus)), key=lambda i: -dense_scores_list[i])
bm25_ranking = sorted(range(len(corpus)), key=lambda i: -bm25_scores[i])

rrf_scores = rrf_scores_of([dense_ranking, bm25_ranking])
hybrid_top1_idx = max(range(len(rrf_scores)), key=lambda i: rrf_scores[i])

print(f'HYBRID (RRF) retrieval:')
ordered = sorted(range(len(rrf_scores)), key=lambda i: -rrf_scores[i])[:3]
for rank, i in enumerate(ordered, 1):
    marker = ' <-- HALCYON doc' if i == TRUE_RELEVANT_IDX else ''
    print(f'  #{rank} [idx={i}, rrf={rrf_scores[i]:.4f}] {corpus[i][:100]}...{marker}')

print()
print(f'Summary so far for query: {failure_query!r}')
print(f'  dense top-1   : idx {dense_top1}           ({"RIGHT" if dense_top1==TRUE_RELEVANT_IDX else "WRONG"})')
print(f'  bm25 top-1    : idx {bm25_top1_idx}           ({"RIGHT" if bm25_top1_idx==TRUE_RELEVANT_IDX else "WRONG"})')
print(f'  hybrid top-1  : idx {hybrid_top1_idx}           ({"RIGHT" if hybrid_top1_idx==TRUE_RELEVANT_IDX else "WRONG"})')

HYBRID (RRF) retrieval:
  #1 [idx=3, rrf=0.0328] All employee badges are managed by the HALCYON access-control system, replaced in 2023 from the prev... <-- HALCYON doc
  #2 [idx=9, rrf=0.0320] Employee expenses go through Concur Expense. Approval chains are based on manager org hierarchy with...
  #3 [idx=0, rrf=0.0318] Our internal CI/CD system is called NIMBUS-OPS. It auto-deploys every merged PR to staging within 90...

Summary so far for query: 'Who provides our employee badge access system?'
  dense top-1   : idx 3           (RIGHT)
  bm25 top-1    : idx 3           (RIGHT)
  hybrid top-1  : idx 3           (RIGHT)


In [10]:
lab.check(3)

OK — hybrid (dense + BM25 via RRF) retrieved relevant doc at top-1
STEP_PASSED


Step 3 Complete! Scroll down to continue...

True

## Step 4 — Cross-encoder reranking

Bi-encoders (our BGE dense embedder) run in two phases: encode docs once at index time, encode query once at query time, compute one matmul. Fast, scales to billions of docs, *but* the query and passage never "meet" inside a model.

**Cross-encoders** concatenate query and passage into a single input, run the whole pair through a transformer, and output one relevance score. Slower (must run per candidate pair, no caching) but dramatically more accurate — they can reason about how specific terms in the query relate to specific parts of the passage.

Production pattern: use bi-encoder (dense) for the first-stage retrieval of ~100 candidates, then cross-encoder to rerank to top 3-5, then send those to the LLM.

[BGE-reranker-base](https://huggingface.co/BAAI/bge-reranker-base) is the standard open cross-encoder (~280 MB). Input: query + passage. Output: a single logit; apply sigmoid for a relevance probability.

In [11]:
from transformers import AutoModelForSequenceClassification

print('Loading BGE-reranker-base (~280 MB first run)...')
RERANK_ID = 'BAAI/bge-reranker-base'
rerank_tokenizer = AutoTokenizer.from_pretrained(RERANK_ID)
reranker = AutoModelForSequenceClassification.from_pretrained(RERANK_ID).to(device)
reranker.eval()

def rerank(query, passages):
    """Score each passage against the query using the cross-encoder. Returns logits (higher = more relevant)."""
    pairs = [(query, p) for p in passages]
    enc = rerank_tokenizer(
        [q for q, _ in pairs], [p for _, p in pairs],
        padding=True, truncation=True, max_length=256, return_tensors='pt',
    ).to(device)
    with torch.no_grad():
        logits = reranker(**enc).logits.squeeze(-1)
    return logits.cpu().tolist()

# Rerank the top-5 hybrid candidates
candidates_idx = sorted(range(len(rrf_scores)), key=lambda i: -rrf_scores[i])[:5]
candidates = [corpus[i] for i in candidates_idx]
rerank_scores_for_candidates = rerank(failure_query, candidates)

# Rebuild a full-corpus score list (non-candidates get -inf so they never win)
rerank_scores = [float('-inf')] * len(corpus)
for orig_idx, rs in zip(candidates_idx, rerank_scores_for_candidates):
    rerank_scores[orig_idx] = rs
reranked_top1_idx = max(range(len(rerank_scores)), key=lambda i: rerank_scores[i])

print(f'\nReranker scores for top-5 hybrid candidates:')
for orig_idx, rs in sorted(zip(candidates_idx, rerank_scores_for_candidates), key=lambda x: -x[1]):
    marker = ' <-- HALCYON doc' if orig_idx == TRUE_RELEVANT_IDX else ''
    print(f'  [idx={orig_idx}, logit={rs:+.3f}] {corpus[orig_idx][:90]}...{marker}')

Loading BGE-reranker-base (~280 MB first run)...


tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]


Reranker scores for top-5 hybrid candidates:
  [idx=3, logit=-1.237] All employee badges are managed by the HALCYON access-control system, replaced in 2023 fro... <-- HALCYON doc
  [idx=1, logit=-5.304] The customer support portal is built on ZephyrDesk v3. It handles 50,000 tickets per day a...
  [idx=9, logit=-7.241] Employee expenses go through Concur Expense. Approval chains are based on manager org hier...
  [idx=0, logit=-10.193] Our internal CI/CD system is called NIMBUS-OPS. It auto-deploys every merged PR to staging...
  [idx=2, logit=-10.197] We use a custom observability stack combining Prometheus, Grafana, and Loki for logs aggre...


In [9]:
lab.check(4)

OK — cross-encoder rerank kept the correct doc at top-1. Top-1 score -1.237, runner-up -5.304 (gap: 4.067)


True

---

## What you just built

A production-shaped retrieval stack: dense bi-encoder + BM25 sparse retriever, fused with RRF, re-ranked by a cross-encoder. This is the *actual* architecture behind every serious enterprise RAG deployment in 2024–2026. You can drop the LLM (Lab 4's step 4) in on top of this and ship it.

### How this changes retrieval precision

- **Dense alone**: good on paraphrased/semantic queries, weak on rare-token queries. Recall@10 typically ~70%.
- **Hybrid (dense + BM25 via RRF)**: recovers the rare-token misses. Recall@10 typically ~85%.
- **Hybrid + cross-encoder rerank**: reorders with pair-aware scoring. Precision@1 typically jumps 20-40 points.

Numbers vary by domain; pattern is universal.

## What to read next

- **[BGE reranker](https://huggingface.co/BAAI/bge-reranker-base)** — the model you just used. Newer v2 variants exist.
- **[Cohere Rerank](https://cohere.com/rerank)** — the commercial state-of-the-art. Often worth the API cost.
- **[Vespa hybrid-search tutorial](https://docs.vespa.ai/en/tutorials/hybrid-search.html)** — production playbook.
- **[BEIR benchmark (Thakur et al., 2021)](https://arxiv.org/abs/2104.08663)** — 18 retrieval datasets where dense-vs-sparse-vs-hybrid is measured head to head.
- **[Advanced RAG patterns (LlamaIndex)](https://docs.llamaindex.ai/en/stable/examples/node_parsers/parser_test/)** — parent-document retrieval, recursive retrieval, auto-merging retrieval.

## What to try next

- **Query rewriting**: use an LLM to rewrite the user's query into 3-5 variants, retrieve for each, fuse with RRF. Measurable gain on conversational queries.
- **Hybrid weighting**: try `w_dense=0.7, w_bm25=0.3` instead of RRF; measure recall@k on your own query set.
- **Swap cross-encoder**: try `BAAI/bge-reranker-large` (~1.1 GB) vs the base variant — do your queries actually benefit?
- **Matryoshka embeddings**: use a model like `mixedbread-ai/mxbai-embed-large-v1` that lets you truncate embedding dimension on the fly — trade retrieval quality for index size.